# Aeolion Geometry Handoff
### Electric V-BAT-Like Tail-Sitter (EDF) — Conceptual Design Study

---

**Purpose.** Emit `out/cad/aeolion_geometry.json` (ADR-0016/0017): the
versioned, SI-unit, solver-native geometry contract for the external
Aeolion VLM/BEMT adjoint loop — planform + CST wing sections, control
surfaces (aileron + jet vanes) with deflection ranges, the fuselage
body of revolution, the propeller (Clark Y sections, rotation sense),
the wing/body placement anchor, and the moment reference point.

Split out of `vehicle_solid_model` (NB8) into its own stage because
this contract grew a design decision of its own (ADR-0017's blade
section) and six independent schema revisions in three days — enough
to earn a dedicated notebook (ADR-0018). Unlike NB8, this one has
**no CadQuery dependency** (`aeolion_handoff.py`/`prop_geometry.py` are
deliberately CadQuery-free) and runs in the local 3.14 venv even when
NB8 is skipped.


---
## 0 — Setup

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # bare-checkout fallback

import conceptual_design as cd
from conceptual_design.notebook import nb_setup
from conceptual_design.design_point import load_handoff, solve_design_point

nb = nb_setup(style=False)   # no matplotlib in this notebook
REPO_ROOT, CONFIG_PATH, OUT_PATH, FIG_DIR = (
    nb.repo_root, nb.config, nb.out, nb.fig_dir)

print("conceptual_design", cd.__version__)

conceptual_design 0.4.2.dev0+g678b361b3.d20260715


---
## 1 — Re-run the sizing loop

Same pattern as every notebook since NB2: reconstruct the converged
design point from `config/` (`design_point.solve_design_point`, the
ADR-0013 single call site) rather than reading a stale snapshot.

In [2]:
inp, result = solve_design_point(CONFIG_PATH)

b_wing_m     = result.wing.b_wing
chord_wing_m = result.wing.chord_mean

print(f"Converged MTOW : {result.m_total_kg:.3f} kg")
print(f"Wing           : b={b_wing_m*1e3:.0f} mm, c={chord_wing_m*1e3:.1f} mm")
print(f"Rotor diameter : {inp.rotor.D_rotor_m*1e3:.0f} mm")

Converged MTOW : 2.518 kg
Wing           : b=1063 mm, c=177.1 mm
Rotor diameter : 203 mm


---
## 2 — Load the upstream handoffs

`out/airfoil.yaml` (NB2), `out/control_vanes.yaml` (NB3),
`out/aileron.yaml` (NB4), `out/vibration.yaml` (NB5, hover 1/rev for
`reference_rpm`), `out/fuselage.yaml` (NB6, body/placement/CG).

In [3]:
af      = load_handoff(OUT_PATH, "airfoil")
vanes   = load_handoff(OUT_PATH, "control_vanes")
ail     = load_handoff(OUT_PATH, "aileron")
vib     = load_handoff(OUT_PATH, "vibration")
fus     = load_handoff(OUT_PATH, "fuselage")

print(f"Airfoil        : {af['designation']}")
print(f"Vanes          : {vanes['n_vanes']} x, delta_max={vanes['delta_max_deg']:.0f} deg")
print(f"Aileron        : delta_max={ail['delta_max_deg']:.0f} deg")
print(f"Hover 1/rev    : {vib['f_shaft_hz']:.1f} Hz")
print(f"Fuselage       : D={fus['D_fus_m']*1e3:.1f} mm, L={fus['L_fus_m']*1e3:.1f} mm")

Airfoil        : NACA 4412
Vanes          : 4 x, delta_max=20 deg
Aileron        : delta_max=20 deg
Hover 1/rev    : 203.5 Hz
Fuselage       : D=98.2 mm, L=491.0 mm


---
## 3 — Propeller blade section and mesh directives

`config/prop_geometry.yaml` (chord/twist/thickness-taper law),
`config/airfoils/clarky.dat` (the Clark Y reference section, ADR-0017),
`config/aeolion.yaml` (fixed station counts / CST order — structural
constants of the optimization, not design variables).

In [4]:
from conceptual_design.aeolion_handoff import AeolionMeshConfig
from conceptual_design.prop_geometry import ClarkYSection, PropGeometry

pg     = PropGeometry.from_yaml(CONFIG_PATH / "prop_geometry.yaml")
clarky = ClarkYSection.from_dat(CONFIG_PATH / "airfoils" / "clarky.dat")
mesh   = AeolionMeshConfig.from_yaml(CONFIG_PATH / "aeolion.yaml")

print(f"Clark Y ref    : t/c={clarky.tc_ref*100:.2f}%, camber_max={clarky.camber_max_ref*100:.2f}%")
print(f"Blade taper    : tc_root={pg.tc_root*100:.0f}% -> tc_tip={pg.tc_tip*100:.0f}%")

Clark Y ref    : t/c=11.71%, camber_max=3.43%


Blade taper    : tc_root=12% -> tc_tip=6%


---
## 4 — Build and export the handoff

In [5]:
from conceptual_design.aeolion_handoff import build_aeolion_geometry, export_aeolion_geometry

aeolion_geometry = build_aeolion_geometry(
    span_m     = b_wing_m,
    chord_m    = chord_wing_m,
    airfoil    = af["designation"],
    ail        = ail,
    vanes      = vanes,
    fus        = fus,
    rotor_D_m  = inp.rotor.D_rotor_m,
    prop       = pg,
    clarky     = clarky,
    f_shaft_hz = vib["f_shaft_hz"],
    mesh       = mesh,
)
path = export_aeolion_geometry(OUT_PATH / "cad" / "aeolion_geometry.json", aeolion_geometry)

print(f"schema         : {aeolion_geometry['schema_version']}")
print(f"design_id      : {aeolion_geometry['design_id'][:26]}...")
print(f"rotation_axis  : {aeolion_geometry['propulsion_bemt']['rotation_axis']}")
size_kb = Path(path).stat().st_size / 1024
print(f"written        : {path}  ({size_kb:,.1f} kB)")

schema         : 1.7.0
design_id      : sha256:150aa29c8e60b5f5ff8...
rotation_axis  : [1.0, 0.0, 0.0]
written        : D:\Dev\vbat-uav-notebooks\out\cad\aeolion_geometry.json  (16.7 kB)


---
## Schema history

See `adr/0016-aeolion-geometry-handoff.md`, `adr/0017-prop-blade-clark-y-section-and-rotation-sense.md`,
and `adr/0018-aeolion-handoff-own-notebook.md` for the full rationale
behind every field in this contract and this notebook split.